In [ ]:

%load_ext autoreload 
%autoreload 2
%matplotlib widget
%matplotlib inline
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
import os
from tqdm.notebook import tqdm
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.lines as mlines 
import sim_loader as sim_load
from scipy.integrate import cumtrapz
from scipy.stats import loguniform


In [ ]:
q_samples = np.geomspace(1e-6, 1e-2, 400)
fig, axs = plt.subplots(1, 1, figsize=(6, 5))
s_minus, s_plus, qs_plot = lensing_triangle_line_QS_space(q_samples, eta=0.35, xi=1/50, Amax=3000)
axs.loglog(s_plus, qs_plot, color='grey', ls='--', label='plus branch')
axs.loglog(s_minus, qs_plot, color='grey',  ls='-',  label='minus branch')
axs.set_xlabel('Projected separation s')
axs.set_ylabel('Mass ratio q')
axs.legend()


## Boxes plot with loguniform distribution in s and q
The idea is to random generate N planets from a loguniform distribution in q and s and the multiplying it by the lensing triangle to yield an occurrence rate and see if it gets close to Romna.
Then on a later step replace the loguniform distribution with my planet distribution

In [ ]:
# Define the grid bins for radius and mass
s_bins = np.geomspace(5e-2, 2e1, 15)  # Log-spaced bins for projected separation  (0.1 to 10 R_E)
q_bins = np.geomspace(1e-6, 1e-2, 15)    # Log-spaced bins for mass ratio (3.3 to 3330 Earth masses)

# create a loguniform distribution of planets in log q and log s
seed = 40
rng = np.random.default_rng(seed)   # pass rng or int to random_state
N = 10000
q_samples = loguniform(1e-6, 1e-2).rvs(size=N, random_state=rng)
s_samples = loguniform(5e-2, 5e1).rvs(size=N, random_state=rng)
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
q_min = xi / Amax

fig, axs = plt.subplots(figsize=(10, 8))

# Create a 2D histogram of planet counts (from the loguniform distribution)
counts, xedges, yedges = np.histogram2d(s_samples, q_samples, bins=[s_bins, q_bins])
# bin centers aligned with counts shape (ns, nq)
s_centers = 0.5*(xedges[:-1] + xedges[1:])
q_centers = 0.5*(yedges[:-1] + yedges[1:])
S, Q = np.meshgrid(s_centers, q_centers, indexing='ij')  # S.shape == counts.shape

# MIGHT HAVE AN ISSUE EXACTLY AT THE TIP IF q is not sampled from 1e-6?
# compute sensitivity mask on bin centers (safe for Q < q_min)
ln_q = np.zeros_like(Q) #starts by filling a vector that will contain the logq with 0s
mask_q = (Q >= q_min) #True where Q >= q_min, False otherwise
print("mask_q shape:", mask_q.shape)
print(mask_q)
ln_q[mask_q] = np.log(Q[mask_q] / q_min) #substitutes the 0s with logq/q_min where mask_q is True
s_plus = np.exp(eta * ln_q) #computation of s_plus and s_minus
s_minus = np.exp(-eta * ln_q)
sens_mask = (Q >= q_min) & (S >= s_minus) & (S <= s_plus) #identifies the inside of the triangle, True inside, False outside
print("sens mask shape:", sens_mask.shape)
# apply sensitivity
detected_counts = counts * sens_mask.astype(float) #multiplies counts times 0 where mask is False and times 1 where mask is True
total_planets = counts.sum()

# percentages per bin (relative to total planets)
percentages = (detected_counts / total_planets) * 100.0

# plot with pcolormesh (use edges; transpose counts to match axes)
mesh = axs.pcolormesh(xedges, yedges, detected_counts.T, cmap='viridis', shading='auto')
cbar = fig.colorbar(mesh, ax=axs)
cbar.set_label('Detected planets (counts * sensitivity)', size = 15)

# annotate percentage (only annotate nonzero bins)
for i in range(s_centers.size):
    for j in range(q_centers.size):
        val = detected_counts[i, j]
        if val > 0:
            txt = f"{int(val)}\n({percentages[i,j]:.1f}%)"
            axs.text(s_centers[i], q_centers[j], txt, ha='center', va='center',
                    color='white', fontsize=8)


#overplot the lensing triangle
s_minus, s_plus, q_out = lensing_triangle_line_QS_space(q_samples, eta, xi, Amax)
axs.loglog(s_plus, q_out, color='grey')
axs.loglog(s_minus, q_out, color='grey')


#  Set axis scales and labels
axs.set_ylim(1e-6, 1e-2)
axs.set_xlim(5e-2, 2e1)
axs.set_xscale('log')
axs.set_yscale('log')
axs.set_xlabel('s [$R_{\mathrm{E}}$]', fontsize=25, labelpad=20)
axs.set_ylabel('q', fontsize=25, labelpad=20)
axs.set_title('Planet counts', fontsize=20)
axs.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
axs.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()
plt.savefig("figures/pop_synth/lensing_triangle", bbox_inches='tight')

## Converting from (q,s) to (m,a) space
The idea is that now we random sample from a loguniform distribution in (m,a) instead of (q,s) and we plot the planet count in (m,a) space

In [ ]:
num = 10000
D_s = 8
D_l = 4
Mp_samples = np.geomspace(1e-3, 1e3, num = num)
Ml_samples = np.geomspace(1e-1*u.M_sun.to(u.M_earth), 1*u.M_sun.to(u.M_earth), num = num)
#Ml_samples = 0.5 * u.M_sun.to(u.M_earth) * np.ones_like(Mp_samples)
q = Mp_samples / Ml_samples
print("shapes:", Mp_samples.shape, Ml_samples.shape, q.shape)
fig, axs = plt.subplots(1, 1, figsize=(6, 5))
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(Mp_samples, Ml_samples, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
print("shapes:", a_plus.shape, Mp_plot.shape) 

axs.loglog(a_plus, Mp_plot, color='grey', ls='--', label='plus branch')
axs.loglog(a_minus, Mp_plot, color='grey',  ls='-',  label='minus branch')
axs.set_xlabel('Semimajor axis a [AU]')
axs.set_ylabel('Planet mass $M_p [M_{\oplus}$]')
axs.legend()
axs.set_xlim(1e-1, 1e2)
axs.set_ylim(1e-3, 1e3)


In [ ]:
N=10000
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
q_min = xi / Amax
D_s = 8
D_l = 4
#lens mass random sample
Ml_samples = np.geomspace(1e-1*u.M_sun.to(u.M_earth), 1*u.M_sun.to(u.M_earth), num = N)
# create a loguniform distribution of planets in log m and log a
a_min = 1e-1
a_max = 1e2
m_min = 1e-3
m_max = 1e3
seed = 2
rng = np.random.default_rng(seed)   # pass rng or int to random_state
m_samples = loguniform(m_min, m_max).rvs(size=N, random_state=rng)
a_samples = loguniform(a_min, a_max).rvs(size=N, random_state=rng)
n_bins = 15
# Define the grid bins for radius and mass
a_bins = np.geomspace(a_min, a_max, n_bins)  # Log-spaced bins for semimajor-axis (1e-1, 1e2) AU
m_bins = np.geomspace(m_min, m_max, n_bins)  # Log-spaced bins for mass  (1e-3, 1e3 Earth masses)

# convert (M,a) to (q,s) per sample
q_samples = m_samples / Ml_samples
s_samples = a_to_s(a_samples, Ml_samples, D_s, D_l)


# flag the planets that are inside the triangles and the ones that are outside
ratio = q_samples / q_min
s_plus = ratio ** eta
s_minus = ratio ** (-eta)
# compute a_minus/a_plus using the matching lens masses
a_minus = s_minus * R_Einstein(Ml_samples, D_s, D_l)
a_plus = s_plus * R_Einstein(Ml_samples, D_s, D_l)
mask = (m_samples >= q_min * Ml_samples) & (a_samples >= a_minus) & (a_samples <= a_plus) # True inside, False outside
sensitivity_mask = mask.astype(float)
n_true = mask.sum()            # number of True
print("Number of detectable planets:", n_true, "out of", N, "total planets.")
fig, ax = plt.subplots(figsize=(10, 8))

# Create a 2D histogram of planet counts (from the loguniform distribution)
count_ma, a_edges, m_edges = np.histogram2d(a_samples, m_samples, bins=[a_bins, m_bins])
# bin centers aligned with counts shape (ns, nq)
a_centers = 0.5*(a_edges[:-1] + a_edges[1:])
m_centers = 0.5*(m_edges[:-1] + m_edges[1:])
# N.B. passing the sensitivity mask as wieghts to histogram2d only works if 0,1 (not even sure it actually does)
# wiegths sums the weight of the samples that fall into each bin (so if a bin has 3 planets, two planets have weight 1,
# one planet has weight 0, the bin will have count 1+1+0=2). If the sensitivity mask is not 0,1, it won't work!!!!!!!
detected_counts, a_edges, m_edges = np.histogram2d(a_samples, m_samples,
                                                   bins=[a_bins, m_bins],
                                                   weights=sensitivity_mask)

# get raw counts for comparison
raw_counts, _, _ = np.histogram2d(a_samples, m_samples, bins=[a_bins, m_bins])
# Verify the detections, sensitivity mask planets and total planets
print("detected",detected_counts.sum(), "sensitivity mask", sensitivity_mask.sum(), "raw", raw_counts.sum())

# plot with pcolormesh (use edges; transpose counts to match axes)
mesh = ax.pcolormesh(a_edges, m_edges, detected_counts.T, cmap='viridis', shading='auto')
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('Detected planets (counts * sensitivity)', size = 20)

# annotate counts / percentages (use bin centers)
for i in range(a_centers.size):
    for j in range(m_centers.size):
        val = detected_counts[i, j]
        raw = raw_counts[i, j]
        if val > 0:
            txt = f"{int(round(val))}\n({int(round(raw))})"
            ax.text(a_centers[i], m_centers[j], txt, ha='center', va='center', color='white', fontsize=8)

# # trying to plot the lensing triangle
# aminus, aplus, m_p = lensing_triangle_line_MA_space(m_samples, Ml_samples, D_s, D_l, eta, xi, Amax)
# ax.loglog(aplus, m_p, color='grey')
# ax.loglog(aminus, m_p, color='grey')

#  Set axis scales and labels
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('a [AU]', fontsize=25, labelpad=20)
ax.set_ylabel('$M_{\mathrm{P}} [M_{\oplus}]$', fontsize=25, labelpad=20)
ax.set_title('Planet counts', fontsize=20)
ax.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
ax.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()
plt.savefig("figures/pop_synth/lensing_triangle_MA_space", bbox_inches='tight')